# From the PDG API to a FastReso decay table

A standalone, runnable companion to the official `tutorial.ipynb`.

The official tutorial shows you how to *read* PDG. This notebook shows what
stands between reading PDG and writing a `decays_*.dat` table for the cascade
code — and it is more than a loop over branching fractions.

**The claim this notebook demonstrates:** the light-flavour table is a *curated
conversion*, not a PDG dump. A generator can reliably decide **which channels
belong**; it cannot invent the numbers PDG does not publish. Everything it
cannot resolve is reported, never guessed.

Run order matters. Restart Kernel and Run All Cells.

## 1. Environment

Only the Python standard library plus the `pdg` package. No numpy, no sympy.

```bash
python -m venv ~/pdg.venv
source ~/pdg.venv/bin/activate
python -m pip install pdg jupyter
```

The curated table is an **input**: it supplies the daughter species and their
quantum numbers, i.e. the set of particles the model is closed over.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import itertools
from collections import Counter, defaultdict
from fractions import Fraction as F
from math import factorial as fac

import pdg

TABLE = Path("../data/azyhydro/decays_QM2016Plus_massorder.dat").resolve()
assert TABLE.is_file(), f"set TABLE to the curated decay list (not found: {TABLE})"

api = pdg.connect()
print(f"pdg package : {version('pdg')}")
print(f"database    : {api}")
print(f"model table : {TABLE.name}")

## 2. The target format

Each species is a **12-field header**:

```
id  name  mass  width  degeneracy  B  S  C  Bot  I  Q  n_channels
2001034  Delta(2950)-  2.95  0.5  16  1  0  0  0  1.5  -1  2
```

followed by exactly `n_channels` **8-field channel rows** with five daughter
slots, zero-padded:

```
id  n_daughters  BR  d1 d2 d3 d4 d5
113  2  1.0  211  -211  0  0  0
```

Points that cost time if you get them wrong:

| | |
|---|---|
| `mass`, `width` | GeV |
| `degeneracy` | spin degeneracy `2J+1`, not `J` |
| `I` | **physical** isospin (`1.5`), not `2I` |
| `Q` | field **11**, *after* `I` — an easy off-by-one |
| `n_channels` | field 12, parsed but **not enforced** by the reader |
| padding | nonzero padding beyond `n_daughters` **is** rejected |
| stable particle | one self-channel `id 1 1 id 0 0 0 0`, never mixed with real rows |

File order ("massorder") is descending mass by convention only; the solver
topologically sorts internally.

## 3. Load the model table

This gives the closure set: every daughter written to the output must appear
here. It also supplies isospin, which the PDG API does not provide in usable
numeric form.

`conserved` is the `(Q, B, S, C, Bot)` vector. Strong and electromagnetic
decays conserve **every** component — summing the whole vector, rather than
charge alone, is what rejects antiparticle combinations later.

`I3` comes from Gell-Mann-Nishijima, `I3 = Q - (B+S+C+Bot)/2`, using only
fields already in the header.

In [ ]:
def read_model(path):
    "Header rows -> {mcid: (name, conserved, I, I3)}, exact Fractions."
    model = {}
    for line in open(path):
        f = line.split()
        if len(f) != 12:
            continue
        B, S, C, Bot = (F(f[i]).limit_denominator(4) for i in (5, 6, 7, 8))
        I, Q = F(f[9]).limit_denominator(4), F(f[10]).limit_denominator(4)
        model[int(f[0])] = (f[1], (Q, B, S, C, Bot), I, Q - F(B + S + C + Bot, 2))
    return model


model = read_model(TABLE)
print(f"{len(model)} species in the model")
for mcid in (2212, 2112, 211, 321, 311, 2214):
    name, cons, I, I3 = model[mcid]
    print(f"  {mcid:>6}  {name:<14} Q,B,S,C,Bot={tuple(map(str, cons))}  I={I}  I3={I3}")

## 4. Isospin: the Clebsch-Gordan coefficient

The strong interaction conserves isospin, so every charge state of a multiplet
decay shares one reduced matrix element. PDG publishes the multiplet rate
(`Delta(1232) --> N pi, 99.4%`); the split among charge states is fixed by
`|<j1 m1 j2 m2 | J M>|^2`.

**PDG gives the split *between* modes. Isospin gives the split *within* a mode.**
You need both sources.

Exact in `Fraction` via the Racah formula — no sympy dependency.

In [ ]:
def cg2(j1, m1, j2, m2, J, M):
    "|<j1 m1 j2 m2 | J M>|^2, exact."
    if m1 + m2 != M or not abs(j1 - j2) <= J <= j1 + j2:
        return F(0)
    if any(abs(m) > j for j, m in ((j1, m1), (j2, m2), (J, M))):
        return F(0)
    pre = (F(2 * J + 1)
           * F(fac(int(J + j1 - j2)) * fac(int(J - j1 + j2)) * fac(int(j1 + j2 - J)),
               fac(int(j1 + j2 + J + 1)))
           * F(fac(int(J + M)) * fac(int(J - M)) * fac(int(j1 - m1))
               * fac(int(j1 + m1)) * fac(int(j2 - m2)) * fac(int(j2 + m2))))
    total = F(0)
    for k in range(200):
        args = [j1 + j2 - J - k, j1 - m1 - k, j2 + m2 - k,
                J - j2 + m1 + k, J - j1 - m2 + k]
        if any(a < 0 or a != int(a) for a in args):
            continue
        d = fac(k)
        for a in args:
            d *= fac(int(a))
        total += F((-1) ** k, d)
    return pre * total * total


half, one, three_half = F(1, 2), F(1), F(3, 2)

# Delta+ (I=3/2, I3=+1/2) -> N pi
assert cg2(half,  half, one, F(0), three_half, half) == F(2, 3)   # p pi0
assert cg2(half, -half, one, one,  three_half, half) == F(1, 3)   # n pi+
# rho0 -> pi0 pi0 vanishes identically (Bose symmetry falls out of the algebra)
assert cg2(one, F(0), one, F(0), one, F(0)) == 0
print("Clebsch-Gordan checks passed:  p pi0 = 2/3,  n pi+ = 1/3,  pi0 pi0 = 0")

## 5. The filter

Four small predicates. Each one answers exactly one question.

**`reject`** — row-level vetoes, in order. Note `is_limit` and `value is None`
are *independent*: a perfectly good non-limit mode can carry no number, and for
`rho` and `Delta` that is precisely the dominant mode. Negative values are real
too — `K+ --> pi+ pi0 gamma (INT)` is an interference term at `-4.24e-06`.

**`expand`** — `PdgItem.particles` (a property, not a method) expands a generic
item into its multiplet members: `N -> {n, p, nbar, pbar}`, `pi -> {pi-, pi0, pi+}`.
The Cartesian product over slots then enumerates every concrete combination.
`item_type` `T`/`I`/`L` (text, inclusive, particle-list) cannot be resolved.

**`usable`** — model closure, no leptons, not purely electromagnetic, 2-4 body,
and full quantum-number conservation.

**`weights`** — `|CG|^2` per surviving combination, merged over orderings of the
same final state. Orthogonality makes these sum to 1 automatically, so no
renormalization step is needed and no rounding drift accumulates.

In [ ]:
LEPTONS = {11, 12, 13, 14, 15, 16}
PHOTON = 22


def reject(bf):
    "Row-level veto. None means the row is usable."
    if list(bf.get_children()): return "group"          # grouping node, not a decay
    if bf.is_limit:             return "limit"
    if bf.value is None:        return "no-value"       # -> human
    if bf.value <= 0:           return "non-positive"   # interference term
    return None


def expand(bf):
    "Every concrete daughter combination, or None if unresolvable."
    slots = []
    for prod in bf.decay_products:
        item = prod.item
        if item.item_type in ("T", "I", "L"):
            return None
        members = [p.mcid for p in item.particles]
        if not members:
            return None
        slots += [members] * int(prod.multiplier or 1)
    return [list(c) for c in itertools.product(*slots)] if slots else None


def usable(mcids, parent, model):
    "Does this concrete combination belong in the table?"
    if any(m not in model for m in mcids):    return False   # model closure
    if any(abs(m) in LEPTONS for m in mcids): return False
    if all(m == PHOTON for m in mcids):       return False
    if not 2 <= len(mcids) <= 4:              return False
    totals = [sum(q) for q in zip(*(model[m][1] for m in mcids))]
    return tuple(totals) == model[parent][1]


def weights(combos, parent, model):
    "Isospin weights, merged over orderings of the same final state."
    _, _, I, I3 = model[parent]
    out, exact = defaultdict(F), True
    for mcids in combos:
        key = tuple(sorted(mcids))
        if len(mcids) == 2:
            (_, _, i1, m1), (_, _, i2, m2) = (model[m] for m in mcids)
            out[key] += cg2(i1, m1, i2, m2, I, I3)
        else:
            exact = False                    # CG here needs a 2-body final state
            out[key] += F(1, len(combos))
    total = sum(out.values())
    return ({k: v / total for k, v in out.items()}, exact) if total else ({}, False)

## 6. The driver

`coverage` is the fraction of the parent's total width the retained channels
account for. It is the single most important number in the report: coverage
well below 1 means the retained rows are **not** a complete decomposition, and
renormalizing them would fabricate branching ratios.

Nothing is written to disk anywhere in this notebook.

In [ ]:
def survey(api, mcid, model, verbose=True):
    parent = model[mcid]
    kept, why, escalate = defaultdict(float), Counter(), []

    for bf in api.get_particle_by_mcid(mcid).exclusive_branching_fractions(
            include_subdecays=True):
        reason = reject(bf)
        if reason:
            why[reason] += 1
            if reason == "no-value":
                escalate.append(bf.description)
            continue

        combos = expand(bf)
        if combos is None:
            why["unresolvable"] += 1
            escalate.append(bf.description)
            continue

        good = [c for c in combos if usable(c, mcid, model)]
        if not good:
            why["filtered"] += 1
            continue

        w, exact = weights(good, mcid, model)
        if not exact:
            why["n-body (no CG)"] += 1
        for key, frac in w.items():
            kept[key] += bf.value * float(frac)

    coverage = sum(kept.values())
    if verbose:
        print(f"=== {parent[0]}  ({mcid})   Q={parent[1][0]}  I={parent[2]}  I3={parent[3]}")
        print(f"    {dict(why)}")
        print(f"    coverage = {coverage:.4f}")
        for key, br in sorted(kept.items(), key=lambda kv: -kv[1]):
            names = " ".join(model[m][0] for m in key)
            print(f"      {br:8.4f}   ({br / coverage:6.4f} renorm)   {names}")
        for d in escalate:
            print(f"      ESCALATE   {d}")
    return kept, why, escalate, coverage

## 7. Run it

Eight species chosen because each fails differently.

In [ ]:
for mcid in (2214, 3224, 323, 213, 113, 221, 223, 333):
    survey(api, mcid, model)
    print()

## 8. Compare with the tracked table

For the two species where isospin does all the work, the generated numbers can
be checked directly against the curated file.

In [ ]:
def tracked(mcid, path=TABLE):
    "Channel rows for one parent, as (BR, (daughters...))."
    rows, seen = [], False
    for line in open(path):
        f = line.split()
        if len(f) == 12:
            seen = int(f[0]) == mcid
        elif len(f) == 8 and seen and int(f[0]) == mcid:
            n = int(f[1])
            rows.append((float(f[2]), tuple(sorted(int(f[3 + i]) for i in range(n)))))
    return rows


for mcid in (2214, 3224):
    kept, _, _, coverage = survey(api, mcid, model, verbose=False)
    gen = {k: v / coverage for k, v in kept.items()}
    print(f"--- {model[mcid][0]} ---")
    print(f"{'channel':<26} {'generated':>10} {'tracked':>9}")
    for br, key in tracked(mcid):
        names = " ".join(model[m][0] for m in key)
        g = gen.get(key)
        print(f"{names:<26} {('%.4f' % g) if g else '     --':>10} {br:9.4f}")
    for key, g in sorted(gen.items(), key=lambda kv: -kv[1]):
        if key not in {k for _, k in tracked(mcid)}:
            names = " ".join(model[m][0] for m in key)
            print(f"{names:<26} {g:10.4f} {'--':>9}   (not in tracked table)")
    print()

`Delta(1232)+` reproduces **exactly**: `2/3` and `1/3`, matching `0.6667` and
`0.3333`. `Sigma(1385)+` matches to rounding, with one extra radiative channel
(`Sigma+ gamma`) that the curated table chose to drop.

That is the generator working. Now the cases where it correctly refuses.

## 9. What cannot be automated

**`rho(770)`** — its dominant mode has no number *and* no concrete daughters:

```
rho(770) --> pi pi      value=None   items: pi(G), pi(G)
rho(770) --> pi+- pi0   value=None   items: pi+-(B), pi0(P)
rho(770) --> pi+- gamma value=0.00045
```

The only quantified row is a 0.045% radiative mode. A generator that filters on
`value is not None` and renormalizes produces `rho -> pi gamma` at **BR 1.0** —
which conserves charge, sums to 1, and passes every automated validation. The
coverage number is the only thing that catches it.

**`K*(892)`** — PDG expresses the dominant mode as `(K pi)+-`, a parenthesised
multiplet that arrives as unresolvable text.

**Branching fractions are stored per isospin multiplet.** `rho+`, `rho0` and
`rho-` all share PDG Identifier `M009` and return byte-identical row sets. The
API object is charge-resolved; its decay table is not. Conservation filtering is
what separates them.

In [ ]:
for mcid in (213, 113, -213):
    p = api.get_particle_by_mcid(mcid)
    rows = [(d.description, d.value) for d in p.exclusive_branching_fractions()]
    print(f"{p.name:<12} pdgid={p.pdgid}  rows={len(rows)}")
ref = [(d.description, d.value) for d in
       api.get_particle_by_mcid(213).exclusive_branching_fractions()]
same = all([(d.description, d.value) for d in
            api.get_particle_by_mcid(m).exclusive_branching_fractions()] == ref
           for m in (113, -213))
print(f"\nidentical row sets across the three charge states: {same}")

## 10. Rendering

Formatting is the easy part, and it is deliberately last: nothing should be
rendered that has not already been reported on. This prints only.

A real generator would additionally decide, explicitly and in writing:

1. which PDG edition to pin, and record it in the file header;
2. whether retained branching ratios stay **absolute** (sum < 1, missing
   probability simply vanishes) or become **conditional** (renormalized to 1);
3. which species are **terminal** — in this model weakly decaying hadrons are
   stable, which is why `K+` carries `width 0` and a self-channel despite having
   29 measured decay modes;
4. how rounding residuals are absorbed so channel sums land where intended.

In [ ]:
def physical(p, kind):
    # Mass or width in GeV, or None. The convenience attributes are not enough:
    # p.mass is None for Delta(1232) because its PDG entry is a MIXED CHARGES
    # summary and the shortcut declines to choose; p.width is None because the
    # width is published as a range ("114 to 120 (~117)") with no single value.
    direct = getattr(p, kind)
    if direct is not None:
        return direct
    for prop in (p.masses() if kind == "mass" else p.widths()):
        if prop.has_best_summary:
            value = prop.best_summary().get_value("GeV")
            if value is not None:
                return value
    return None


for mcid in (213, 2214, 333):
    p = api.get_particle_by_mcid(mcid)
    print(f"{p.name:<14} attr mass={str(p.mass):<10} -> {physical(p, 'mass')}"
          f"   attr width={str(p.width):<10} -> {physical(p, 'width')}")

A missing width is **not** a harmless gap. Rendered as `0` it would make
`Delta(1232)` a stable particle: `width 0` plus a self-channel is exactly how
this format encodes terminal species. The whole cascade would change, and no
validator would object. So the renderer refuses rather than defaulting.

In [ ]:
def render(mcid, model, kept, coverage, conditional=True, overrides=None):
    "12-field header plus 8-field channel rows. Returns text; writes nothing."
    name, (Q, B, S, C, Bot), I, _ = model[mcid]
    p = api.get_particle_by_mcid(mcid)
    overrides = overrides or {}
    mass = overrides.get("mass", physical(p, "mass"))
    width = overrides.get("width", physical(p, "width"))
    if mass is None or width is None:
        missing = "mass" if mass is None else "width"
        raise ValueError(
            f"{name}: no {missing} from PDG. Supply it explicitly — defaulting "
            f"a width to 0 would silently make this species terminal."
        )
    if not p.quantum_J:
        raise ValueError(f"{name}: no J, cannot form the 2J+1 degeneracy")

    # Fractions are right for the arithmetic and wrong for the file: Julia's
    # reader calls parse(Float64, field), and parse(Float64, "3/2") raises.
    num = lambda x: f"{float(x):g}"

    deg = 2 * F(p.quantum_J) + 1
    scale = coverage if conditional and coverage else 1.0
    rows = sorted(kept.items(), key=lambda kv: -kv[1])

    out = ["\t".join([str(mcid), name, f"{mass:.4g}", f"{width:.4g}", num(deg),
                      num(B), num(S), num(C), num(Bot), num(I), num(Q),
                      str(len(rows))])]
    for key, br in rows:
        daughters = list(key) + [0] * (5 - len(key))
        out.append("\t".join([str(mcid), str(len(key)), f"{br / scale:.4f}",
                              *map(str, daughters)]))
    return "\n".join(out)


kept, _, _, coverage = survey(api, 2214, model, verbose=False)

try:
    render(2214, model, kept, coverage)
except ValueError as error:
    print(f"refused: {error}\n")

# PDG publishes the Delta width as the range "114 to 120 (~117)" MeV. Taking
# 0.117 GeV is a documented human decision, not something the API handed over.
print("generated (width supplied by hand):")
print(render(2214, model, kept, coverage, overrides={"width": 0.117}))
print("\ntracked:")
print("2214\tDelta(1232)+" + open(TABLE).read()
      .split("2214\tDelta(1232)+")[1].split("\n2114")[0])

## 11. Where this leaves you

The filter reliably answers **which channels belong in the table**. It cannot
supply numbers PDG does not publish, and it should not pretend to.

Remaining work, in descending order of value:

1. **Terminal predicate** — decide stability from quark content before any decay
   logic runs. It short-circuits every weakly decaying hadron and is the single
   highest-value rule.
2. **K0L/K0S -> K0/K0bar** — `phi`'s second-largest mode (0.336) is lost to model
   closure without it.
3. **Double-count detection** — group retained rows by sorted daughter tuple and
   flag collisions such as `pi+ pi- pi0` versus `(pi+ pi- pi0) S-wave`, or
   `pi0 gamma gamma` versus its non-resonant subset.
4. **Parenthesised multiplets** — `(K pi)+-` and friends.
5. **Curated input** for every species still escalating.

Validate any candidate on the Julia side before trusting it:

```julia
run(`julia --project=. tools/convert_decay_data.jl azyhydro candidate.dat --particles-out candidate_particles.dat --decays-out candidate_decays.dat`)
using FastReso2
dl = DecayList(readparticles("candidate_particles.dat"), readdecays("candidate_decays.dat"))
validate(dl)          # prints a report and returns stats; it does NOT throw
```

`validate` checks charge, baryon number, strangeness and charm conservation,
branching-ratio ranges and sums, daughter closure, and kinematic thresholds.
It reports; reading the numbers is your job.